# Xbar-S Analysis

Xbar-S charts are the gold standard for monitoring processes with subgrouped data. They're ideal when you have:

- Multiple measurements per time period
- Rational subgroups (factors like machines, operators, batches)
- Need to monitor both location (mean) and spread (variation)

## What You'll Learn

1. Create Xbar and S charts from replicated data
2. Understand how SDS affects variance estimation
3. Compare factor levels using control charts
4. Access VAS residuals for deeper analysis

## Setup

In [1]:
import numpy as np
import pandas as pd
from processbehavior import ProcessBehavior

## Create Replicated Data

We'll simulate a filling machine with:
- 3 operators (A, B, C)
- 8 time periods
- 4 replicate measurements per operator per time period

This creates **SDS 1: Full Replication** - the most powerful design.

In [2]:
np.random.seed(42)

operators = ['A', 'B', 'C']
n_times = 8
n_reps = 4

data = []
for t in range(n_times):
    for op in operators:
        # Each operator has a slightly different mean
        op_effect = {'A': 0, 'B': 2, 'C': -1}[op]
        
        # Add time trend (process drift)
        time_effect = t * 0.3
        
        for rep in range(n_reps):
            # Add special cause for Operator B at time 6
            special = 8 if (op == 'B' and t == 6) else 0
            
            value = 100 + op_effect + time_effect + special + np.random.normal(0, 1.5)
            data.append({
                'time': t + 1,
                'operator': op,
                'weight': round(value, 2)
            })

df = pd.DataFrame(data)
print(f"Dataset: {len(df)} observations")
print(f"Structure: {len(operators)} operators x {n_times} times x {n_reps} reps")
df.head(12)

Dataset: 96 observations
Structure: 3 operators x 8 times x 4 reps


,time,operator,weight
0,1,A,100.75
1,1,A,99.79
2,1,A,100.97
3,1,A,102.28
4,1,B,101.65
5,1,B,101.65
6,1,B,104.37
7,1,B,103.15
8,1,C,98.30
9,1,C,99.81


## Formulate the Study

In [3]:
pb = ProcessBehavior(df)

study = pb.formulate(
    response=pb.cols.weight,
    factors=[pb.cols.operator],
    time=pb.cols.time
)

print(f"SDS: {study.sds} ({study.sds_name})")
print(f"Description: {study.sds_description}")
print(f"\nValid charts: {study.valid_charts}")
print(f"Recommended: {study.recommended_chart}")
print(f"Residual charts: {study.residual_charts}")

SDS: 1 (Full Factorial with Complete Replication)
Description: All factor × time cells have n ≥ 2 observations (best case for analysis)

Valid charts: ['Xbar', 'S', 'R', 'Imr']
Recommended: Xbar
Residual charts: ['R2_S', 'R3_Xbar', 'R3_S', 'R4_Xbar', 'R4_S', 'R5_Xbar', 'R5_S']


## Understanding SDS 1

**SDS 1 (Full Replication)** is the most powerful sampling design because:

1. Every (operator, time) cell has multiple observations
2. Within-cell variance can be estimated exactly
3. All VAS residuals (R1-R5) are available
4. Interactions can be detected

The formula for control limits uses the pooled within-cell standard deviation.

## Execute Xbar-S Charts Analysis

In [4]:
result = study.execute()  # Uses recommended chart (Xbar)

print(f"Charts created: {result.all_charts}")
print(f"Has residuals: {result.has_residuals}")

Charts created: ['Xbar', 'S']
Has residuals: True


## View Chart Data

In [5]:
# Xbar chart shows subgroup means
xbar_data = result.get_chart('Xbar')
print("Xbar Chart Data (subgroup means):")
xbar_data.head(10)

Xbar Chart Data (subgroup means):


,rsg,xbar,center,lpl,upl,beyond_limits
0,A,100.567,101.549,100.452,102.646,0
1,B,104.222,101.549,100.452,102.646,1
2,C,99.859,101.549,100.452,102.646,-1


In [6]:
# S chart shows subgroup standard deviations
s_data = result.get_chart('S')
print("\nS Chart Data (subgroup std devs):")
s_data.head(10)


S Chart Data (subgroup std devs):


,rsg,s,center,lpl,upl,beyond_limits
0,A,1.669,2.052,1.267,2.837,0
1,B,3.017,2.052,1.267,2.837,1
2,C,1.470,2.052,1.267,2.837,0


In [7]:
# Statistics for both charts
print("Xbar Statistics:")
display(result.get_statistics('Xbar'))

print("\nSbar Statistics:")
display(result.get_statistics('S'))

Xbar Statistics:


{'center': np.float64(101.549),
 'N': np.int64(32),
 'upl': np.float64(102.646),
 'lpl': np.float64(100.452)}


Sbar Statistics:


{'center': np.float64(2.052),
 'N': np.int64(32),
 'upl': np.float64(2.837),
 'lpl': np.float64(1.267)}

## Visualize Xbar Chart

In [8]:
fig = result.plot(
    chart='Xbar',
    show_zones=True,
    highlight_signals=True,
    show_stats=True
)
fig.show()

## Visualize S Chart

In [9]:
fig = result.plot(
    chart='S',
    show_zones=True,
    highlight_signals=True
)
fig.show()

## Understanding Xbar-S Charts

### The Xbar Chart

- Plots the **mean** of each subgroup
- Centerline: Grand mean of all observations
- Limits based on within-subgroup variation
- Detects **shifts in process level**

### The S Chart

- Plots the **standard deviation** of each subgroup
- Centerline: Pooled within-subgroup standard deviation
- Limits based on chi-square distribution
- Detects **changes in process variation**

### Reading Order

1. **First check the S chart** - Variation must be stable
2. **Then interpret the Xbar chart** - Only valid if S is stable
3. Points on Xbar beyond limits → investigate the specific subgroup

## Signal Detection for Xbar-S

For Xbar and S charts (categorical comparisons), only **Rule 1** applies - points beyond the control limits.

In [10]:
# Detect signals on Xbar
signals = result.detect_signals(chart='Xbar')

print(f"Xbar signals: {signals.count}")
if signals.has_signals:
    print("\nViolations:")
    display(signals.violations)

Xbar signals: 2

Violations:


,obs_id,rule_name,rule_number,description,value,center,upl,lpl
0,1,rule_1,1,Point beyond control limits,104.222,101.549,102.646,100.452
1,2,rule_1,1,Point beyond control limits,99.859,101.549,102.646,100.452


In [11]:
# Detect signals on Sbar
signals_s = result.detect_signals(chart='S')

print(f"Sbar chart signals: {signals_s.count}")

Sbar chart signals: 1


## Accessing VAS Residuals

With SDS 1 (full replication), all five residuals are available:

In [12]:
# View the computed residuals
residuals = result.residuals
print("VAS Residuals:")
residuals.head(10)

VAS Residuals:


,R1,R2,R3,R4,R5
0,-0.799479,-0.1975,0.955625,-0.970313,-1.179792
1,-1.759479,-1.1575,-0.004375,-1.930313,-2.139792
2,-0.579479,0.0225,1.175625,-0.750313,-0.959792
3,0.730521,1.3325,2.485625,0.559687,0.350208
12,-0.889479,1.8450,1.896458,0.041354,0.862708
13,-4.119479,-1.3850,-1.333542,-3.188646,-2.367292
14,-3.839479,-1.1050,-1.053542,-2.908646,-2.087292
15,-2.089479,0.6450,0.696458,-1.158646,-0.337292
24,-1.769479,-0.3650,0.119792,-1.271979,-1.347292
25,-0.779479,0.6250,1.109792,-0.281979,-0.357292


In [ ]:
# Analyze time effects using R4 residuals on S chart
result_r4 = study.execute(chart='S', value='R4')

# View the chart data
print("R4 Residual on S Chart (Time Effects):")
print(f"Charts: {result_r4.all_charts}")
result_r4.get_chart('S').head()

In [ ]:
# Analyze factor (operator) effects using R5 residuals on S chart
result_r5 = study.execute(chart='S', value='R5')

# View the chart data
print("R5 Residual on S Chart (Operator Effects):")
print(f"Charts: {result_r5.all_charts}")
result_r5.get_chart('S').head()

## Chart Table Summary

Get a compact summary table for reporting:

In [15]:
# Summary table with subgroup info, values, and limits
table = result.chart_table('Xbar')
table

,subgroup,n,value,center,lpl,upl,signal
0,A,32,100.567,101.549,100.452,102.646,
1,B,32,104.222,101.549,100.452,102.646,↑
2,C,32,99.859,101.549,100.452,102.646,↓


## Summary

In this tutorial, you learned:

- Xbar-S charts require subgrouped data (n >= 2 per cell)
- SDS 1 (full replication) provides the most analytical power
- The S chart monitors variation; the Xbar chart monitors level
- Only Rule 1 applies to Xbar-S charts
- VAS residuals enable deeper root cause analysis

## Next Steps

- [Stratified Analysis](stratified-analysis.ipynb) - Separate charts per factor level
- [VAS Residuals](../user-guide/residuals.md) - Deep dive into VAS residuals
- [Signal Detection](signal-detection.ipynb) - All Western Electric rules